# Solution Comparison Dashboard

Compare two `output_payload` JSON solutions side-by-side.

- **Solution Overview** — key metrics table with delta columns
- **Gantt charts** — driver timelines for both solutions on a shared Y axis
- **Distance per service** — side-by-side stacked bars
- **Distance per driver** — side-by-side stacked bars
- **Summary metrics** — KPI comparison with delta columns (requires eval reports)

All logic lives in `src/analysis/compare_solutions.py`. Edit the **Configuration** cell only.

> **Baseline note**: solutions without `addData` (historic snapshots) have `duration_min=0`;
> their VEHICLE_TRANSPORTATION bars are rendered with a 5-minute minimum width as position markers.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
from typing import Optional

_PROJECT_ROOT = Path("..").resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from src.analysis.compare_solutions import (
    load_and_prepare,
    build_overview_table,
    build_comparison_gantt_figure,
    build_comparison_service_figure,
    build_comparison_driver_figure,
    build_comparison_summary_table,
    build_comparison_route_map,
)
from src.optimization.settings.model_params import ModelParams
from src.optimization.settings.solver_settings import DEFAULT_DISTANCE_METHOD

In [2]:
# ── Configuration ──────────────────────────────────────────────────────────────────────

SOL_A_PAYLOAD = Path("../experiments/phase2/cases/file_snapshots/output_payload_scoped.json")
SOL_B_PAYLOAD = Path("../experiments/phase2/results/output/output_payload.json")
INPUT_FILE: Optional[Path] = Path("../experiments/phase2/cases/input_mirror.json")
DRIVER_DIRECTORY: Optional[Path] = Path("../data/model_input/driver_directory.json")

EVAL_A: Optional[Path] = None
EVAL_B: Optional[Path] = None

LABEL_A = "baseline"
LABEL_B = "algorithm"

PLANNING_DATE: Optional[str] = "2026-02-18"
DISTANCE_METHOD: str = DEFAULT_DISTANCE_METHOD

_params = ModelParams()
ALFRED_SPEED_KMH: float = _params.alfred_speed_kmh
print(f"alfred_speed_kmh={ALFRED_SPEED_KMH} | distance_method={DISTANCE_METHOD!r}")

alfred_speed_kmh=30.0 | distance_method='osrm'


In [3]:
# ── Load and prepare both solutions ──────────────────────────────────────────────────
pair = load_and_prepare(
    sol_a=SOL_A_PAYLOAD,
    sol_b=SOL_B_PAYLOAD,
    input_file=INPUT_FILE,
    driver_directory=DRIVER_DIRECTORY,
    planning_date=PLANNING_DATE,
    distance_method=DISTANCE_METHOD,
    speed_kmh=ALFRED_SPEED_KMH,
)

[driver_home_lookup] 13 drivers loaded
[coord_lookup] 57 labors | method='osrm'
sol_a: 44 labors | 12 drivers | 99 segments
sol_b: 44 labors | 11 drivers | 97 segments


---
## Solution Overview

In [4]:
build_overview_table(pair, LABEL_A, LABEL_B)

,baseline,algorithm,delta,delta_pct
metric,,,,
services,35.00,35.00,0.00,0.00
labors_total,44.00,44.00,0.00,0.00
labors_vt,39.00,39.00,0.00,0.00
labors_non_vt,5.00,5.00,0.00,0.00
drivers_used,12.00,11.00,-1.00,-8.33
labors_assigned,39.00,39.00,0.00,0.00
labors_unassigned,5.00,5.00,0.00,0.00
labors_infeasible,0.00,0.00,0.00,NaN
labors_timeline_overlap,2.00,0.00,-2.00,-100.00


---
## Gantt Charts — Driver Timelines

In [5]:
build_comparison_gantt_figure(pair, LABEL_A, LABEL_B).show()

---
## Distance per Service

In [6]:
build_comparison_service_figure(pair, LABEL_A, LABEL_B).show()

---
## Distance per Driver

In [6]:
build_comparison_driver_figure(pair, LABEL_A, LABEL_B).show()

---
## Summary Metrics
_Requires `EVAL_A` / `EVAL_B` to be set._

In [8]:
tbl = build_comparison_summary_table(EVAL_A, EVAL_B, LABEL_A, LABEL_B)
if tbl is not None:
    display(tbl)

No evaluation reports provided — set EVAL_A / EVAL_B to enable this section.


---
## Route Comparison Map

Select a driver from the dropdown to view both solutions' routes side by side.

Colour legend:
- **Blue / cadetblue** — baseline (service labor / driver move)
- **Red / orange** — algorithm (service labor / driver move)
- Dashed lines indicate straight-line fallback when OSRM is unavailable.

In [7]:
import ipywidgets as widgets
from IPython.display import display
from src.analysis.solution_evaluation import build_route_map

_COLORS_A = {"VEHICLE_TRANSPORTATION": "blue",   "DRIVER_MOVE": "cadetblue"}
_COLORS_B = {"VEHICLE_TRANSPORTATION": "red",    "DRIVER_MOVE": "orange"}

# Pixel dimensions for each map — adjust to taste.
_MAP_W, _MAP_H = 640, 520

_drv_dropdown = widgets.Dropdown(
    options=pair.all_drivers,
    description="Driver:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)

_out_a = widgets.Output(layout=widgets.Layout(width=f"{_MAP_W}px"))
_out_b = widgets.Output(layout=widgets.Layout(width=f"{_MAP_W}px"))

_map_row = widgets.HBox(
    [
        widgets.VBox(
            [widgets.HTML(f"<h4 style='text-align:center;margin:4px 0'>{LABEL_A}</h4>"), _out_a],
        ),
        widgets.VBox(
            [widgets.HTML(f"<h4 style='text-align:center;margin:4px 0'>{LABEL_B}</h4>"), _out_b],
        ),
    ],
)


def _render_maps(driver_id):
    home_wkt = (pair.driver_home_lookup or {}).get(str(driver_id))

    _out_a.clear_output(wait=True)
    with _out_a:
        try:
            display(build_route_map(
                services=None,
                rows=pair.rows_a,
                driver_id=driver_id,
                driver_home_wkt=home_wkt,
                label=LABEL_A,
                points_lookup=pair.points_lookup,
                colors=_COLORS_A,
                map_width=_MAP_W,
                map_height=_MAP_H,
            ))
        except Exception as e:
            print(f"{e}")

    _out_b.clear_output(wait=True)
    with _out_b:
        try:
            display(build_route_map(
                services=None,
                rows=pair.rows_b,
                driver_id=driver_id,
                driver_home_wkt=home_wkt,
                label=LABEL_B,
                points_lookup=pair.points_lookup,
                colors=_COLORS_B,
                map_width=_MAP_W,
                map_height=_MAP_H,
            ))
        except Exception as e:
            print(f"{e}")


_drv_dropdown.observe(lambda change: _render_maps(change["new"]), names="value")
_render_maps(_drv_dropdown.value)
display(_drv_dropdown, _map_row)

Dropdown(description='Driver:', layout=Layout(width='300px'), options=('10451', '10500', '11714', '11988', '14…